# P0A-2 Customer Churn Predictor
### DevJam AI Engineer Roadmap - Machine Learning Foundations

This notebook provides a complete hands-on walkthrough of a supervised machine learning classification pipeline. We will explore user behaviors, contract lengths, billing metrics, and support tickets to understand and predict **Customer Churn**.

### Learning Objectives:
1. **Binary Classification**: What it is and why churn fits this definition.
2. **Exploratory Data Analysis (EDA)**: Visualizing distributions and correlation heatmaps to discover churn drivers.
3. **Data Preprocessing**: Standard scaling, One-Hot Encoding, and preventing data leakage.
4. **Supervised ML Models**: Training and comparing Logistic Regression, Decision Trees, Random Forests, and Gradient Boosting.
5. **Evaluation Metrics**: Going beyond Accuracy to explore Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 1. Load the Dataset
We load our generated customer dataset, inspect its shape, columns, and check for any missing values.

In [ ]:
# Load data
data_path = '../data/churn_data.csv'
if not os.path.exists(data_path):
    # Fallback to local run folder if running from notebooks dir directly
    data_path = 'data/churn_data.csv'

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Check schema and missing values
df.info()

## 2. Exploratory Data Analysis (EDA)
Let's look at key drivers of customer churn.

In [ ]:
# Churn Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='churn', data=df, palette='Set2')
plt.title('Overall Customer Churn Count')
plt.show()

In [ ]:
# Tenure vs Churn
plt.figure(figsize=(8, 5))
sns.boxplot(x='churn', y='tenure_months', data=df, palette='Set1')
plt.title('Customer Tenure (Months) vs. Churn')
plt.show()

In [ ]:
# Monthly Charges vs Churn
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='monthly_charges', hue='churn', fill=True, common_norm=False, palette='Set1', alpha=0.5)
plt.title('Monthly Charges Distribution by Churn Status')
plt.show()

In [ ]:
# Support Tickets vs Churn
plt.figure(figsize=(10, 5))
sns.countplot(x='support_tickets', hue='churn', data=df, palette='Set2')
plt.title('Churn count by filed Support Tickets')
plt.show()

In [ ]:
# Correlation Heatmap for numeric values
plt.figure(figsize=(8, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numeric Customer Features')
plt.show()

## 3. Data Preprocessing
We separate target and features, set up scaling for numerical features and One-Hot Encoding for categorical inputs, and perform the train-test split.

In [ ]:
# Drop unique IDs and extract target
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn'].map({'Yes': 1, 'No': 0})

# Define column groups
numerical_features = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 
                      'support_tickets', 'last_login_days', 'usage_frequency']
categorical_features = ['gender', 'contract_type', 'payment_method', 'internet_service', 
                        'plan_type', 'has_premium_support']

# Create transformer pipelines
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_features),
    ('cat', cat_transformer, categorical_features)
])

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Reconstruct Feature Names
cat_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features).tolist()
all_features = numerical_features + cat_names

print(f"Training set shape: {X_train_processed.shape}")
print(f"Test set shape: {X_test_processed.shape}")

## 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=8),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1, max_depth=4)
}

results = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

results_df = pd.DataFrame(results).T
results_df

## 5. Model Feature Importances
Let's see which customer features drive the classification decisions in our Random Forest classifier.

In [ ]:
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices[:12]], y=np.array(all_features)[indices[:12]], palette='viridis')
plt.title('Top 12 Features - Random Forest Importance')
plt.xlabel('Importance')
plt.show()

## 6. Confusion Matrix & Analysis
Let's look at the predictions of our best classifier.

In [ ]:
# Find best model name by F1 Score
best_model_name = results_df['F1 Score'].idxmax()
best_model = models[best_model_name]

y_pred_best = best_model.predict(X_test_processed)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Retained', 'Churned'], 
            yticklabels=['Retained', 'Churned'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_test, y_pred_best))